# 01 — Business Understanding, Data Gathering, Cleaning & EDA

Notebook ini mendokumentasikan tahapan awal proyek dari pemahaman masalah hingga eksplorasi data. Dataset portofolio tidak memuat `plat_nomor`. Seluruh hasil adalah analisis snapshot periode 2025, bukan pengukuran kesiapan operasional real-time.


## 1. Business Understanding

### Latar belakang masalah
Dinas penanggulangan kebakaran dan penyelamatan memerlukan gambaran inventaris kendaraan yang tercatat dalam kondisi **BAIK** atau **RUSAK**. Ringkasan kondisi dapat membantu mengidentifikasi pola pencatatan aset dan kelompok yang perlu ditinjau lebih lanjut. Dataset ini hanya merupakan snapshot inventaris periode 2025; label kondisi bukan ukuran kesiapan operasional real-time.

### Pernyataan masalah
Belum tersedia ringkasan analitis yang mudah dibandingkan mengenai distribusi kondisi kendaraan menurut wilayah administratif, jenis kendaraan, klasifikasi, fungsi, dan penempatan. Analisis ini bertujuan memberikan gambaran deskriptif dan menguji kelayakan awal klasifikasi label kondisi yang tercatat.

### Pertanyaan analisis
1. Bagaimana distribusi label kondisi BAIK dan RUSAK pada data?
2. Apakah proporsi kondisi RUSAK berbeda di antara wilayah dan jenis kendaraan?
3. Variabel kategorikal apa yang menunjukkan asosiasi eksploratif dengan label kondisi?
4. Apakah fitur yang tersedia cukup untuk eksperimen klasifikasi baseline, dan apa risiko bias atau kebocoran targetnya?

### Tujuan dan manfaat
- Memeriksa kualitas serta struktur data inventaris kendaraan.
- Menyajikan distribusi kondisi dan profil administratif dengan jumlah serta persentase.
- Menyiapkan dataset secara transparan untuk eksperimen baseline, tanpa menjadikan hasil model sebagai keputusan operasional.
- Mendokumentasikan keterbatasan data dan kebutuhan informasi tambahan.

### Metode dan metrik
Tahap ini menggunakan analisis deskriptif, tabulasi silang, visualisasi, serta asosiasi kategorikal (Cramér's V sebagai ukuran eksploratif, bukan kausalitas). Untuk eksperimen klasifikasi lanjutan, target adalah `Kondisi`; model dibandingkan dengan baseline mayoritas. Metrik yang diprioritaskan ialah **macro-F1**, **balanced accuracy**, **recall kelas RUSAK**, precision kelas RUSAK, dan confusion matrix karena label tidak seimbang. Hasil harus dibaca sebagai eksperimen pada data tercatat, bukan prediksi kesiapan operasional.

### Batasan dan risiko
- Data hanya memuat periode 2025; tidak dapat digunakan untuk menyimpulkan tren atau memprediksi perubahan dari waktu ke waktu.
- Tidak tersedia koordinat, riwayat perawatan, waktu respons, ketersediaan kru, atau beban kejadian.
- Label kondisi dan proses pencatatannya tidak dijelaskan lebih rinci dalam file yang tersedia.
- Kategori penempatan/wilayah mungkin berkaitan erat dengan pola pencatatan dan berpotensi menyebabkan model mempelajari pola administratif, bukan kondisi teknis.
- Kolom `plat_nomor` dikeluarkan dari dataset portofolio karena merupakan identifier aset.
- Lisensi/ketentuan redistribusi harus diverifikasi pada halaman sumber; jangan mengasumsikan CC0.


## 2. Data Gathering dan pemahaman dataset

**Sumber resmi:** [Satu Data Jakarta — Data Kendaraan Operasional Penanggulangan Kebakaran dan Penyelamatan](https://satudata.jakarta.go.id/open-data/detail?kategori=dataset&page_url=data-kendaraan-operasional-penanggulangan-kebakaran-dan-penyelamatan&data_no=1)

Data yang dianalisis adalah inventaris kendaraan operasional penanggulangan kebakaran dan penyelamatan DKI Jakarta dengan periode data 2025. File sumber yang diperiksa memiliki 1.153 baris dan 10 kolom; kolom `plat_nomor` tidak disertakan dalam salinan portofolio untuk mengurangi paparan identifier. Karena itu, file olahan dalam repository berisi 1.153 baris dan 9 kolom.

**Alasan pemilihan:** data berkaitan langsung dengan kebutuhan memahami kondisi inventaris kendaraan, memiliki lebih dari 100 baris dan lebih dari 4 fitur selain target/identifier, serta memuat variasi label kondisi dan isu kualitas yang dapat diperiksa. Kelayakan akhir tetap bergantung pada validasi sumber, deskripsi kolom, dan ketentuan penggunaan.

### Data dictionary

| Kolom | Makna berdasarkan nama/isi data | Peran dalam analisis |
|---|---|---|
| `periode_data` | Tahun/periode data | Konteks waktu; konstan 2025 sehingga dikeluarkan dari fitur model |
| `wilayah` | Wilayah administratif | Profil deskriptif; kandidat fitur dengan evaluasi risiko bias |
| `kecamatan` | Kecamatan | Profil administratif; kandidat fitur |
| `kelurahan` | Kelurahan | Profil administratif; kandidat fitur |
| `jenis_kendaraan` | Jenis/nama kendaraan | Profil kendaraan; kandidat fitur |
| `klasifikasi` | Klasifikasi kendaraan | Profil kendaraan; kandidat fitur |
| `fungsi` | Fungsi kendaraan yang tercatat | Profil fungsi; kandidat fitur dengan pemeriksaan potensi leakage |
| `penempatan` | Lokasi/instansi penempatan yang tercatat | Profil penempatan; kandidat fitur berisiko menangkap pola pencatatan |
| `Kondisi` | Label kondisi tercatat, misalnya BAIK/RUSAK | Target klasifikasi |

Makna di atas ditafsirkan dari nama kolom dan nilai yang tersedia, bukan pengganti metadata resmi. Jika portal menyediakan definisi formal, gunakan definisi portal tersebut.

### Kredibilitas, privasi, dan lisensi
- Sumber berasal dari portal resmi Satu Data Jakarta; verifikasi kembali instansi pengampu, tanggal pembaruan, deskripsi variabel, dan lisensi pada halaman sumber.
- File sumber yang diperiksa menggunakan ekstensi `.xls`, tetapi dapat terbaca sebagai tabel HTML. Salinan mentah yang memuat `plat_nomor` tidak disertakan dalam repository publik.
- Jangan menyatakan lisensi CC0 atau izin redistribusi tanpa bukti eksplisit. Jika lisensi tidak jelas, tautkan sumber resmi dan patuhi ketentuan portal.
- Dataset portofolio adalah salinan sanitasi; jumlah dan nilai kategorinya sebaiknya diverifikasi terhadap sumber resmi sebelum publikasi.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "processed" / "kendaraan_2025_sanitized.csv"
df = pd.read_csv(DATA_PATH)
df.head()

## 3. Cleaning: pemeriksaan dan penanganan kualitas data

Cleaning dilakukan berdasarkan audit data, bukan dengan menghapus baris secara otomatis. Karena identifier `plat_nomor` telah dikeluarkan, dua baris yang identik pada sembilan kolom tersisa **belum tentu** merupakan duplikasi aset: kendaraan berbeda dapat memiliki atribut administratif dan kondisi yang sama. Oleh sebab itu, duplikasi pada salinan sanitasi dilaporkan sebagai pemeriksaan, tetapi tidak langsung dihapus tanpa identifier yang aman atau konfirmasi sumber.

Penanganan yang diterapkan:
1. Simpan salinan `df_raw` agar data hasil baca tidak tertimpa.
2. Standarkan spasi pada kolom teks dan ubah string kosong menjadi missing value.
3. Audit missing value, label target, dan duplikasi sebelum/sesudah.
4. Jangan menghapus baris hanya karena seluruh atribut sanitasi sama. Missing pada fitur akan ditangani dalam pipeline preprocessing; baris tanpa target tidak digunakan untuk klasifikasi.
5. Dokumentasikan perubahan jumlah baris. Cleaning ini tidak mengarang nilai dan tidak menyimpulkan nilai yang tidak tersedia.


In [ ]:
df_raw = df.copy()

def audit_ringkas(data, tahap):
    return {
        "tahap": tahap,
        "jumlah_baris": len(data),
        "jumlah_kolom": data.shape[1],
        "sel_missing": int(data.isna().sum().sum()),
        "baris_dengan_missing": int(data.isna().any(axis=1).sum()),
        "baris_identik_di_salinan_sanitasi": int(data.duplicated().sum()),
        "target_missing": int(data["Kondisi"].isna().sum()) if "Kondisi" in data else None
    }

sebelum = audit_ringkas(df_raw, "Sebelum cleaning")
df_clean = df_raw.copy()

# Rapikan spasi pada kolom teks dan samakan string kosong menjadi missing.
text_cols = df_clean.select_dtypes(include=["object", "string"]).columns
for col in text_cols:
    df_clean[col] = df_clean[col].astype("string").str.strip()
    df_clean[col] = df_clean[col].replace("", pd.NA)

sesudah = audit_ringkas(df_clean, "Sesudah cleaning")
display(pd.DataFrame([sebelum, sesudah]).set_index("tahap"))

print("Missing per kolom setelah cleaning:")
display(df_clean.isna().sum().rename("jumlah_missing").to_frame())

print(
    "Catatan duplikasi: jumlah ini dihitung pada kolom yang tersedia setelah identifier "
    "dihapus. Baris tidak dihapus otomatis karena bisa mewakili kendaraan berbeda."
)

# Gunakan df_clean untuk analisis berikutnya.
df = df_clean.copy()


In [ ]:
print("Dimensi:", df.shape)
print("\nTipe data:")
display(df.dtypes.to_frame("dtype"))
print("\nMissing value:")
display(df.isna().sum().rename("missing").to_frame())
print("\nDuplikasi identik:", df.duplicated().sum())
print("\nPeriode:")
display(df["periode_data"].value_counts(dropna=False))
print("\nDistribusi target:")
display(pd.concat([
    df["Kondisi"].value_counts(dropna=False).rename("jumlah"),
    (df["Kondisi"].value_counts(normalize=True, dropna=False)*100).round(2).rename("persen")
], axis=1))

## Pemeriksaan kualitas dan kategori

Dataset bersifat dominan kategorikal. Karena itu, deteksi outlier numerik konvensional seperti IQR tidak menjadi fokus. Audit dilakukan terhadap nilai kosong, duplikasi, kategori, dan konsistensi hierarki administratif.

In [ ]:
for col in df.columns:
    print(f"{col}: {df[col].nunique(dropna=True)} kategori unik")
print("\nNilai kategori target:", sorted(df["Kondisi"].dropna().unique().tolist()))
print("\nKonsistensi label Kondisi:")
display(df["Kondisi"].value_counts(dropna=False))

In [ ]:
target_counts = df["Kondisi"].value_counts()
ax = target_counts.reindex(["BAIK", "RUSAK"]).plot(kind="bar", rot=0)
ax.set_title("Distribusi Kondisi Kendaraan")
ax.set_xlabel("Kondisi tercatat")
ax.set_ylabel("Jumlah entri")
plt.tight_layout()
plt.show()
print("Rasio BAIK terhadap RUSAK:", round(target_counts.get("BAIK", 0)/target_counts.get("RUSAK", 1), 2))

In [ ]:
wilayah = pd.crosstab(df["wilayah"], df["Kondisi"])
wilayah["TOTAL"] = wilayah.sum(axis=1)
wilayah["RUSAK_PCT"] = (wilayah.get("RUSAK", 0) / wilayah["TOTAL"] * 100).round(2)
display(wilayah.sort_values("RUSAK_PCT", ascending=False))
plot_data = wilayah.sort_values("RUSAK_PCT", ascending=False)
ax = plot_data["RUSAK_PCT"].plot(kind="bar", rot=70)
ax.set_title("Persentase RUSAK menurut Wilayah")
ax.set_ylabel("RUSAK (%)")
plt.tight_layout()
plt.show()

In [ ]:
jenis = pd.crosstab(df["jenis_kendaraan"], df["Kondisi"])
jenis["TOTAL"] = jenis.sum(axis=1)
jenis["RUSAK_PCT"] = (jenis.get("RUSAK", 0) / jenis["TOTAL"] * 100).round(2)
display(jenis.sort_values(["RUSAK_PCT", "TOTAL"], ascending=[False, False]).head(15))
display(df["jenis_kendaraan"].value_counts().head(10).rename("jumlah").to_frame())

## Asosiasi kategorikal

Cramér’s V mengukur kekuatan asosiasi antara dua variabel kategorikal, bukan sebab-akibat. Banyaknya kategori dan sel dengan frekuensi harapan kecil dapat memengaruhi interpretasi. Nilai p dari chi-square hanya ditampilkan sebagai eksplorasi.

In [ ]:
def cramers_v(table):
    chi2 = chi2_contingency(table, correction=False)[0]
    n = table.to_numpy().sum()
    r, k = table.shape
    if n == 0 or min(r, k) <= 1:
        return np.nan
    return np.sqrt(chi2 / (n * min(k-1, r-1)))

rows = []
for col in ["wilayah", "kecamatan", "kelurahan", "jenis_kendaraan", "klasifikasi", "fungsi", "penempatan"]:
    tab = pd.crosstab(df[col], df["Kondisi"])
    chi2, p, dof, expected = chi2_contingency(tab, correction=False)
    rows.append({"variabel": col, "kategori": df[col].nunique(),
                 "cramers_v": cramers_v(tab), "chi2_p_eksploratif": p,
                 "sel_expected_lt5_pct": (expected < 5).mean()*100})
assoc = pd.DataFrame(rows).sort_values("cramers_v", ascending=False)
display(assoc.round(4))

## Interpretasi awal

- Kelas BAIK dominan; akurasi saja tidak cukup untuk menilai model.
- Proporsi RUSAK perlu dibaca bersama jumlah entri pada setiap kelompok.
- `plat_nomor` tidak tersedia dalam data portofolio sanitasi dan tidak boleh digunakan sebagai prediktor default.
- Tanpa koordinat, analisis per wilayah administratif tidak dapat disebut klasterisasi spasial berbasis jarak.
- Hasil asosiasi tidak membuktikan bahwa lokasi atau jenis kendaraan menyebabkan kerusakan.